In [2]:
!pip install -q huggingface_hub safetensors einops
!pip install -q causal-conv1d mamba-ssm --no-build-isolation

In [3]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print(torch.version.cuda)

True
Tesla T4
12.8


In [4]:
import os

if not os.path.exists("/kaggle/working/BioFoundation"):
    !git clone https://github.com/pulp-bio/BioFoundation.git /kaggle/working/BioFoundation

Cloning into '/kaggle/working/BioFoundation'...
remote: Enumerating objects: 652, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 652 (delta 66), reused 44 (delta 34), pack-reused 532 (from 1)
Receiving objects: 100% (652/652), 2.90 MiB | 24.32 MiB/s, done.
Resolving deltas: 100% (211/211), done.


In [5]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="PulpBio/FEMBA",
    repo_type="model",
    local_dir="/kaggle/working/checkpoints/FEMBA"
)

for root, dirs, files in os.walk("/kaggle/working/checkpoints/FEMBA"):
    for file in files:
        if file.endswith(".safetensors"):
            print(os.path.join(root, file))

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

/kaggle/working/checkpoints/FEMBA/TUSL/FEMBA_base.safetensors
/kaggle/working/checkpoints/FEMBA/TUSL/FEMBA_large.safetensors
/kaggle/working/checkpoints/FEMBA/TUSL/FEMBA_tiny.safetensors
/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_base.safetensors
/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_large.safetensors
/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_tiny.safetensors
/kaggle/working/checkpoints/FEMBA/TUAB/FEMBA_base.safetensors
/kaggle/working/checkpoints/FEMBA/TUAB/FEMBA_large.safetensors


In [6]:
import os
import re
import sys
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from safetensors.torch import load_file

sys.path.insert(0, "/kaggle/working/BioFoundation")

from models.FEMBA import FEMBA

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

cuda
Tesla T4


In [7]:
femba_checkpoint_path = "/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_tiny.safetensors"

print(os.path.exists(femba_checkpoint_path))
print(femba_checkpoint_path)

state = load_file(femba_checkpoint_path)

a_log_shape = None

for k, v in state.items():
    if k == "mamba_blocks.0.mamba_fwd.A_log":
        a_log_shape = v.shape
        print(k, v.shape)

num_channels = 22
patch_channel = 2
channel_patches = num_channels // patch_channel
expand = 4

embed_dim = a_log_shape[0] // (channel_patches * expand)

block_ids = []

for k in state.keys():
    m = re.match(r"mamba_blocks\.(\d+)\.", k)
    if m:
        block_ids.append(int(m.group(1)))

num_blocks = max(block_ids) + 1

print("embed_dim:", embed_dim)
print("num_blocks:", num_blocks)

True
/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_tiny.safetensors
mamba_blocks.0.mamba_fwd.A_log torch.Size([1540, 16])
embed_dim: 35
num_blocks: 2


In [8]:
class FEMBA_ECG_ABP(nn.Module):
    def __init__(
        self,
        seq_length=625,
        num_channels=22,
        output_len=625,
        embed_dim=35,
        num_blocks=2
    ):
        super().__init__()

        self.num_channels = num_channels

        self.femba = FEMBA(
            seq_length=seq_length,
            num_channels=num_channels,
            num_classes=1,
            patch_size=(2, 16),
            stride=(2, 16),
            embed_dim=embed_dim,
            num_blocks=num_blocks,
            classification_type="bc"
        )

        self.femba = self.femba.to(device)

        dummy = torch.zeros(1, num_channels, seq_length, device=device)
        mask = torch.zeros_like(dummy, dtype=torch.bool, device=device)

        self.femba.eval()

        with torch.no_grad():
            features = self.encode(dummy, mask)

        feature_dim = features.shape[-1]

        self.regression_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, output_len)
        )

    def encode(self, x, mask):
        x_masked = x.clone()
        x_masked[mask] = 0

        x = self.femba.patch_embed(x_masked)
        x = x + self.femba.pos_embed

        for mamba_block, norm_layer in zip(self.femba.mamba_blocks, self.femba.norm_layers):
            res = x
            x = mamba_block(x)
            x = res + x
            x = norm_layer(x)

        x = x.mean(dim=1)

        return x

    def forward(self, x):
        if x.ndim == 2:
            x = x.unsqueeze(1)

        if x.shape[1] == 1:
            x = x.repeat(1, self.num_channels, 1)

        x = x.to(device)
        mask = torch.zeros_like(x, dtype=torch.bool, device=device)

        features = self.encode(x, mask)
        y = self.regression_head(features)

        return y.unsqueeze(1)

In [9]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model_femba_tuar_tiny = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
).to(device)

print("Model created")

Model created


In [10]:
model_dict = model_femba_tuar_tiny.state_dict()
matched_state = {}

for ckpt_key, ckpt_value in state.items():
    candidates = [
        ckpt_key,
        "femba." + ckpt_key
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == ckpt_value.shape:
            matched_state[candidate] = ckpt_value
            break

missing, unexpected = model_femba_tuar_tiny.load_state_dict(matched_state, strict=False)

print("Loaded tensors:", len(matched_state))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

for k in list(matched_state.keys())[:20]:
    print(k, matched_state[k].shape)

Loaded tensors: 42
Missing keys: 18
Unexpected keys: 0
femba.mamba_blocks.0.mamba_fwd.A_log torch.Size([1540, 16])
femba.mamba_blocks.0.mamba_fwd.D torch.Size([1540])
femba.mamba_blocks.0.mamba_fwd.conv1d.bias torch.Size([1540])
femba.mamba_blocks.0.mamba_fwd.conv1d.weight torch.Size([1540, 1, 4])
femba.mamba_blocks.0.mamba_fwd.dt_proj.bias torch.Size([1540])
femba.mamba_blocks.0.mamba_fwd.dt_proj.weight torch.Size([1540, 25])
femba.mamba_blocks.0.mamba_fwd.in_proj.weight torch.Size([3080, 385])
femba.mamba_blocks.0.mamba_fwd.out_proj.weight torch.Size([385, 1540])
femba.mamba_blocks.0.mamba_fwd.x_proj.weight torch.Size([57, 1540])
femba.mamba_blocks.0.mamba_rev.A_log torch.Size([1540, 16])
femba.mamba_blocks.0.mamba_rev.D torch.Size([1540])
femba.mamba_blocks.0.mamba_rev.conv1d.bias torch.Size([1540])
femba.mamba_blocks.0.mamba_rev.conv1d.weight torch.Size([1540, 1, 4])
femba.mamba_blocks.0.mamba_rev.dt_proj.bias torch.Size([1540])
femba.mamba_blocks.0.mamba_rev.dt_proj.weight torch.S

In [12]:
import sys
import torch
import platform
import subprocess
import psutil

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU memory:", round(props.total_memory / 1024**3, 2), "GB")
else:
    print("GPU: No GPU")

ram_gb = psutil.virtual_memory().total / 1024**3
print("RAM:", round(ram_gb, 2), "GB")

try:
    nvcc = subprocess.check_output(["nvcc", "--version"]).decode()
    print("\nNVCC:")
    print(nvcc)
except Exception as e:
    print("\nNVCC not found or unavailable")

Python: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.10.0+cu128
PyTorch CUDA: 12.8
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
RAM: 31.35 GB

NVCC:
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0



In [14]:
import gdown
import numpy as np

# For intra-subject

def download_dataset(file_id):
    download_url = f"https://drive.google.com/uc?id={file_id}"
    output_file = "mimic.npy"
    gdown.download(download_url, output_file, quiet=False)
    data = np.load("mimic.npy", allow_pickle=True).item()
    return data

def load_dataset(dataset):
    X = []  # ECG segments
    y = []  # ABP segments

    # Loop over each patient
    for patient_id, signals in dataset.items():
        ecg_segments = signals['ecg']
        abp_segments = signals['abp']
        # Safety check: skip if segment lengths don't match
        if len(ecg_segments) != len(abp_segments):
            continue

        for ecg_seg, abp_seg in zip(ecg_segments, abp_segments):
            if (
                isinstance(ecg_seg, np.ndarray) and ecg_seg.shape == (625,) and
                isinstance(abp_seg, np.ndarray) and abp_seg.shape == (625,)
            ):
                X.append(ecg_seg)
                y.append(abp_seg)


    # Convert lists to numpy arrays
    X = np.array(X)
    y = np.array(y)
    
    return X, y


links = {
    # this is all from mimic-iii dataset
    "no_pp": "18nRE4Z_RBKjT3grYFUiCV0cjUw6gZNd5",
    "bw": "1mui4ifqbZv9e-5GeE4XyU061Kxi-wZdN",
    "bw+maf": "1wRU1UTPOzVqa_2lIm8GLI94jeLdf4pgh",
    "dwt": "1HYGwjSBrgiMV-4CSOiMxJXFyNzlSDBq8",
    "nk2": "1ZrELLKXvmhsuPhirAcvsZ4TcXtRC1rGQ",

    # this is from mimic-iv
    "no_pp_m4":"1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY",
    "bw_m4":"1tK1fPxBhZicWW5Vd5S8DFrKefnHQFVjQ",
    "bw+maf_m4":"12KmtjeVHuZP4omk-AvEBpfwmI1cPn1R7",
    "nk2_m4":"12AHW7_3ytaBs34hGHsIJ-ruRC8vpNE9q",
    "dwt_m4":"1Mmb9NxnwvKBSa1dmNJtAcF63V8N8BxYR"
  }
data = download_dataset(links["no_pp_m4"])
X,y = load_dataset(data)
print(X.shape, y.shape)

Downloading...
From (original): https://drive.google.com/uc?id=1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY
From (redirected): https://drive.google.com/uc?id=1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY&confirm=t&uuid=b9d7c9ee-d661-40e1-9f92-4ec34426aa10
To: /kaggle/working/mimic.npy
100%|██████████| 384M/384M [00:06<00:00, 56.1MB/s] 


(25319, 625) (25319, 625)


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [20]:
from sklearn.model_selection import train_test_split

X_ecg = X
y_abp = y

X_train, X_test, y_train, y_test = train_test_split(
    X_ecg,
    y_abp,
    test_size=0.2,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=SEED
)

print("Raw train:", X_train.shape, y_train.shape)
print("Raw val:", X_val.shape, y_val.shape)
print("Raw test:", X_test.shape, y_test.shape)

Raw train: (16204, 625) (16204, 625)
Raw val: (4051, 625) (4051, 625)
Raw test: (5064, 625) (5064, 625)


In [21]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(
    X_train.reshape(-1, 1)
).reshape(X_train.shape)

X_val_scaled = X_scaler.transform(
    X_val.reshape(-1, 1)
).reshape(X_val.shape)

X_test_scaled = X_scaler.transform(
    X_test.reshape(-1, 1)
).reshape(X_test.shape)

y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
).reshape(y_train.shape)

y_val_scaled = y_scaler.transform(
    y_val.reshape(-1, 1)
).reshape(y_val.shape)

y_test_scaled = y_scaler.transform(
    y_test.reshape(-1, 1)
).reshape(y_test.shape)

In [22]:
X_train_scaled = torch.tensor(X_train_scaled, dtype=torch.float32).unsqueeze(1)
X_val_scaled = torch.tensor(X_val_scaled, dtype=torch.float32).unsqueeze(1)
X_test_scaled = torch.tensor(X_test_scaled, dtype=torch.float32).unsqueeze(1)

y_train_scaled = torch.tensor(y_train_scaled, dtype=torch.float32).unsqueeze(1)
y_val_scaled = torch.tensor(y_val_scaled, dtype=torch.float32).unsqueeze(1)
y_test_scaled = torch.tensor(y_test_scaled, dtype=torch.float32).unsqueeze(1)

print("Train:", X_train_scaled.shape, y_train_scaled.shape)
print("Val:", X_val_scaled.shape, y_val_scaled.shape)
print("Test:", X_test_scaled.shape, y_test_scaled.shape)

Train: torch.Size([16204, 1, 625]) torch.Size([16204, 1, 625])
Val: torch.Size([4051, 1, 625]) torch.Size([4051, 1, 625])
Test: torch.Size([5064, 1, 625]) torch.Size([5064, 1, 625])


In [19]:
import os
import random
import numpy as np
import torch

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print(e)

print("Seed fixed:", SEED)

Seed fixed: 42


In [23]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 8

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    TensorDataset(X_train_scaled, y_train_scaled),
    batch_size=batch_size,
    shuffle=True,
    generator=g
)

val_loader = DataLoader(
    TensorDataset(X_val_scaled, y_val_scaled),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test_scaled, y_test_scaled),
    batch_size=batch_size,
    shuffle=False
)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

train batches: 2026
val batches: 507
test batches: 633


In [24]:
model_femba_tuar_tiny.eval()

with torch.no_grad():
    xb, yb = next(iter(train_loader))
    xb = xb.to(device).float()
    yb = yb.to(device).float()
    out = model_femba_tuar_tiny(xb)

print("Input:", xb.shape)
print("Output:", out.shape)
print("Target:", yb.shape)

Input: torch.Size([8, 1, 625])
Output: torch.Size([8, 1, 625])
Target: torch.Size([8, 1, 625])


In [26]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()

            pred = model(xb)
            loss = criterion(pred, yb)

            total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)

In [27]:
criterion_femba_tuar_tiny = nn.MSELoss()

optimizer_femba_tuar_tiny_full = torch.optim.AdamW(
    model_femba_tuar_tiny.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

num_epochs_femba_tuar_tiny_full = 30
best_val_loss_femba_tuar_tiny_full = float("inf")

best_model_path_femba_tuar_tiny_full = "femba_tuar_tiny_full_ft_ecg_abp_seed42.pth"
history_femba_tuar_tiny_full = []

for epoch in range(num_epochs_femba_tuar_tiny_full):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny,
        train_loader,
        optimizer_femba_tuar_tiny_full,
        criterion_femba_tuar_tiny,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny,
        val_loader,
        criterion_femba_tuar_tiny,
        device
    )

    history_femba_tuar_tiny_full.append({
        "method": "FEMBA_TUAR_tiny_full_finetuning",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 1e-4,
        "weight_decay": 1e-2,
        "seed": SEED
    })

    if val_loss < best_val_loss_femba_tuar_tiny_full:
        best_val_loss_femba_tuar_tiny_full = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": "FEMBA_TUAR_tiny_full_finetuning",
            "model_state_dict": model_femba_tuar_tiny.state_dict(),
            "optimizer_state_dict": optimizer_femba_tuar_tiny_full.state_dict(),
            "best_val_loss": float(best_val_loss_femba_tuar_tiny_full),
            "lr": 1e-4,
            "weight_decay": 1e-2,
            "seed": SEED
        }, best_model_path_femba_tuar_tiny_full)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_femba_tuar_tiny_full}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Epoch [1/30] | Train Loss: 0.974233 | Val Loss: 0.961482 saved
Epoch [2/30] | Train Loss: 0.929950 | Val Loss: 0.928664 saved
Epoch [3/30] | Train Loss: 0.903333 | Val Loss: 0.911173 saved
Epoch [4/30] | Train Loss: 0.873098 | Val Loss: 0.876251 saved
Epoch [5/30] | Train Loss: 0.818113 | Val Loss: 0.802798 saved
Epoch [6/30] | Train Loss: 0.739918 | Val Loss: 0.730698 saved
Epoch [7/30] | Train Loss: 0.661245 | Val Loss: 0.668522 saved
Epoch [8/30] | Train Loss: 0.592005 | Val Loss: 0.605695 saved
Epoch [9/30] | Train Loss: 0.532009 | Val Loss: 0.561653 saved
Epoch [10/30] | Train Loss: 0.482587 | Val Loss: 0.531215 saved
Epoch [11/30] | Train Loss: 0.440716 | Val Loss: 0.500575 saved
Epoch [12/30] | Train Loss: 0.403853 | Val Loss: 0.472748 saved
Epoch [13/30] | Train Loss: 0.374637 | Val Loss: 0.451632 saved
Epoch [14/30] | Train Loss: 0.348551 | Val Loss: 0.436971 saved
Epoch [15/30] | Train Loss: 0.326624 | Val Loss: 0.420408 saved
Epoch [16/30] | Train Loss: 0.308429 | Val Loss: 

In [28]:
import pandas as pd

history_femba_tuar_tiny_full_df = pd.DataFrame(history_femba_tuar_tiny_full)
history_femba_tuar_tiny_full_df.to_csv("history_femba_tuar_tiny_full_ft.csv", index=False)

history_femba_tuar_tiny_full_df.tail()

,method,epoch,train_loss,val_loss,lr,weight_decay,seed
25,FEMBA_TUAR_tiny_full_finetuning,26,0.210766,0.359494,0.0001,0.01,42
26,FEMBA_TUAR_tiny_full_finetuning,27,0.206406,0.349608,0.0001,0.01,42
27,FEMBA_TUAR_tiny_full_finetuning,28,0.199974,0.346266,0.0001,0.01,42
28,FEMBA_TUAR_tiny_full_finetuning,29,0.196517,0.342568,0.0001,0.01,42
29,FEMBA_TUAR_tiny_full_finetuning,30,0.192647,0.339336,0.0001,0.01,42


In [29]:
checkpoint_femba_tuar_tiny_full = torch.load(
    best_model_path_femba_tuar_tiny_full,
    map_location="cpu"
)

model_femba_tuar_tiny_best = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
)

model_femba_tuar_tiny_best.load_state_dict(
    checkpoint_femba_tuar_tiny_full["model_state_dict"]
)

model_femba_tuar_tiny_best = model_femba_tuar_tiny_best.to(device)
model_femba_tuar_tiny_best.eval()

print("Loaded best FEMBA TUAR tiny full fine-tuned model")
print("Best epoch:", checkpoint_femba_tuar_tiny_full["epoch"])
print("Best val loss:", checkpoint_femba_tuar_tiny_full["best_val_loss"])

Loaded best FEMBA TUAR tiny full fine-tuned model
Best epoch: 30
Best val loss: 0.3393355619168758


In [30]:
def evaluate_model_waveform(model, loader, y_scaler, device):
    model.eval()

    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device).float()

            pred = model(xb)

            pred = pred.detach().cpu().numpy()
            true = yb.detach().cpu().numpy()

            pred = pred.reshape(pred.shape[0], -1)
            true = true.reshape(true.shape[0], -1)

            y_pred_all.append(pred)
            y_true_all.append(true)

    y_pred_all = np.concatenate(y_pred_all, axis=0)
    y_true_all = np.concatenate(y_true_all, axis=0)

    y_pred_real = y_scaler.inverse_transform(
        y_pred_all.reshape(-1, 1)
    ).reshape(y_pred_all.shape)

    y_true_real = y_scaler.inverse_transform(
        y_true_all.reshape(-1, 1)
    ).reshape(y_true_all.shape)

    return y_true_real, y_pred_real

In [31]:
y_true_femba_tuar_tiny_full, y_pred_femba_tuar_tiny_full = evaluate_model_waveform(
    model_femba_tuar_tiny_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_femba_tuar_tiny_full.shape)
print("y_pred:", y_pred_femba_tuar_tiny_full.shape)

print("true min/max:", y_true_femba_tuar_tiny_full.min(), y_true_femba_tuar_tiny_full.max())
print("pred min/max:", y_pred_femba_tuar_tiny_full.min(), y_pred_femba_tuar_tiny_full.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 37.62044 192.60255


In [32]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def extract_sbp_dbp(signal):
    sbp = np.max(signal)
    dbp = np.min(signal)
    return sbp, dbp

sbp_true_femba_tuar_tiny_full = []
dbp_true_femba_tuar_tiny_full = []
sbp_pred_femba_tuar_tiny_full = []
dbp_pred_femba_tuar_tiny_full = []

for true_sig, pred_sig in zip(y_true_femba_tuar_tiny_full, y_pred_femba_tuar_tiny_full):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_femba_tuar_tiny_full.append(sbp_t)
    dbp_true_femba_tuar_tiny_full.append(dbp_t)
    sbp_pred_femba_tuar_tiny_full.append(sbp_p)
    dbp_pred_femba_tuar_tiny_full.append(dbp_p)

sbp_true_femba_tuar_tiny_full = np.array(sbp_true_femba_tuar_tiny_full)
dbp_true_femba_tuar_tiny_full = np.array(dbp_true_femba_tuar_tiny_full)
sbp_pred_femba_tuar_tiny_full = np.array(sbp_pred_femba_tuar_tiny_full)
dbp_pred_femba_tuar_tiny_full = np.array(dbp_pred_femba_tuar_tiny_full)

sbp_mae_femba_tuar_tiny_full = mean_absolute_error(
    sbp_true_femba_tuar_tiny_full,
    sbp_pred_femba_tuar_tiny_full
)

dbp_mae_femba_tuar_tiny_full = mean_absolute_error(
    dbp_true_femba_tuar_tiny_full,
    dbp_pred_femba_tuar_tiny_full
)

sbp_rmse_femba_tuar_tiny_full = np.sqrt(mean_squared_error(
    sbp_true_femba_tuar_tiny_full,
    sbp_pred_femba_tuar_tiny_full
))

dbp_rmse_femba_tuar_tiny_full = np.sqrt(mean_squared_error(
    dbp_true_femba_tuar_tiny_full,
    dbp_pred_femba_tuar_tiny_full
))

sbp_r2_femba_tuar_tiny_full = r2_score(
    sbp_true_femba_tuar_tiny_full,
    sbp_pred_femba_tuar_tiny_full
)

dbp_r2_femba_tuar_tiny_full = r2_score(
    dbp_true_femba_tuar_tiny_full,
    dbp_pred_femba_tuar_tiny_full
)

avg_mae_femba_tuar_tiny_full = (
    sbp_mae_femba_tuar_tiny_full + dbp_mae_femba_tuar_tiny_full
) / 2

print("FEMBA TUAR Tiny Full Fine-Tuning Results")
print("========================================")

print("SBP")
print(f"MAE  : {sbp_mae_femba_tuar_tiny_full:.3f}")
print(f"RMSE : {sbp_rmse_femba_tuar_tiny_full:.3f}")
print(f"R2   : {sbp_r2_femba_tuar_tiny_full:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_femba_tuar_tiny_full:.3f}")
print(f"RMSE : {dbp_rmse_femba_tuar_tiny_full:.3f}")
print(f"R2   : {dbp_r2_femba_tuar_tiny_full:.4f}")

print("\nTable format:")
print(f"{sbp_mae_femba_tuar_tiny_full:.2f}/{dbp_mae_femba_tuar_tiny_full:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_femba_tuar_tiny_full:.3f}")

FEMBA TUAR Tiny Full Fine-Tuning Results
SBP
MAE  : 10.532
RMSE : 14.068
R2   : 0.1056

DBP
MAE  : 4.819
RMSE : 6.601
R2   : 0.1267

Table format:
10.53/4.82

Average MAE:
7.675


In [33]:
results_femba_tuar_tiny_full = pd.DataFrame([{
    "model": "FEMBA",
    "checkpoint": "TUAR tiny",
    "fine_tuning": "full fine-tuning",
    "loss": "MSELoss",
    "seed": SEED,
    "best_epoch": checkpoint_femba_tuar_tiny_full["epoch"],
    "best_val_loss": checkpoint_femba_tuar_tiny_full["best_val_loss"],
    "sbp_mae": sbp_mae_femba_tuar_tiny_full,
    "dbp_mae": dbp_mae_femba_tuar_tiny_full,
    "avg_mae": avg_mae_femba_tuar_tiny_full,
    "sbp_rmse": sbp_rmse_femba_tuar_tiny_full,
    "dbp_rmse": dbp_rmse_femba_tuar_tiny_full,
    "sbp_r2": sbp_r2_femba_tuar_tiny_full,
    "dbp_r2": dbp_r2_femba_tuar_tiny_full
}])

results_femba_tuar_tiny_full.to_csv(
    "results_femba_tuar_tiny_full_ft.csv",
    index=False
)

results_femba_tuar_tiny_full

,model,checkpoint,fine_tuning,loss,seed,best_epoch,best_val_loss,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,FEMBA,TUAR tiny,full fine-tuning,MSELoss,42,30,0.339336,10.532134,4.81866,7.675397,14.068002,6.601086,0.105582,0.126704


# Loss other

In [34]:
class WaveformSBPDBPLoss(nn.Module):
    def __init__(self, waveform_weight=1.0, sbp_weight=0.5, dbp_weight=0.5):
        super().__init__()
        self.waveform_weight = waveform_weight
        self.sbp_weight = sbp_weight
        self.dbp_weight = dbp_weight
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        waveform_loss = self.mse(pred, target)

        pred_sbp = torch.max(pred, dim=-1).values
        target_sbp = torch.max(target, dim=-1).values

        pred_dbp = torch.min(pred, dim=-1).values
        target_dbp = torch.min(target, dim=-1).values

        sbp_loss = self.mse(pred_sbp, target_sbp)
        dbp_loss = self.mse(pred_dbp, target_dbp)

        total_loss = (
            self.waveform_weight * waveform_loss
            + self.sbp_weight * sbp_loss
            + self.dbp_weight * dbp_loss
        )

        return total_loss

In [35]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model_femba_tuar_tiny_sbpdbp = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
).to(device)

model_dict = model_femba_tuar_tiny_sbpdbp.state_dict()
matched_state = {}

for ckpt_key, ckpt_value in state.items():
    candidates = [
        ckpt_key,
        "femba." + ckpt_key
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == ckpt_value.shape:
            matched_state[candidate] = ckpt_value
            break

missing, unexpected = model_femba_tuar_tiny_sbpdbp.load_state_dict(matched_state, strict=False)

print("Loaded tensors:", len(matched_state))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Loaded tensors: 42
Missing keys: 18
Unexpected keys: 0


In [36]:
criterion_femba_tuar_tiny_sbpdbp = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_femba_tuar_tiny_sbpdbp = torch.optim.AdamW(
    model_femba_tuar_tiny_sbpdbp.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

In [37]:
num_epochs_femba_tuar_tiny_sbpdbp = 30
best_val_loss_femba_tuar_tiny_sbpdbp = float("inf")

best_model_path_femba_tuar_tiny_sbpdbp = "femba_tuar_tiny_full_ft_sbpdbp_loss_seed42.pth"
history_femba_tuar_tiny_sbpdbp = []

for epoch in range(num_epochs_femba_tuar_tiny_sbpdbp):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny_sbpdbp,
        train_loader,
        optimizer_femba_tuar_tiny_sbpdbp,
        criterion_femba_tuar_tiny_sbpdbp,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny_sbpdbp,
        val_loader,
        criterion_femba_tuar_tiny_sbpdbp,
        device
    )

    history_femba_tuar_tiny_sbpdbp.append({
        "method": "FEMBA_TUAR_tiny_full_finetuning_sbpdbp_loss",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 1e-4,
        "weight_decay": 1e-2,
        "seed": SEED,
        "waveform_weight": 1.0,
        "sbp_weight": 0.5,
        "dbp_weight": 0.5
    })

    if val_loss < best_val_loss_femba_tuar_tiny_sbpdbp:
        best_val_loss_femba_tuar_tiny_sbpdbp = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": "FEMBA_TUAR_tiny_full_finetuning_sbpdbp_loss",
            "model_state_dict": model_femba_tuar_tiny_sbpdbp.state_dict(),
            "optimizer_state_dict": optimizer_femba_tuar_tiny_sbpdbp.state_dict(),
            "best_val_loss": float(best_val_loss_femba_tuar_tiny_sbpdbp),
            "lr": 1e-4,
            "weight_decay": 1e-2,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_femba_tuar_tiny_sbpdbp)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_femba_tuar_tiny_sbpdbp}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Epoch [1/30] | Train Loss: 1.270174 | Val Loss: 1.272805 saved
Epoch [2/30] | Train Loss: 1.087838 | Val Loss: 1.086653 saved
Epoch [3/30] | Train Loss: 1.029318 | Val Loss: 1.055163 saved
Epoch [4/30] | Train Loss: 0.995785 | Val Loss: 1.039447 saved
Epoch [5/30] | Train Loss: 0.967038 | Val Loss: 1.038869 saved
Epoch [6/30] | Train Loss: 0.946087 | Val Loss: 1.025189 saved
Epoch [7/30] | Train Loss: 0.930131 | Val Loss: 1.012148 saved
Epoch [8/30] | Train Loss: 0.913667 | Val Loss: 1.008775 saved
Epoch [9/30] | Train Loss: 0.901811 | Val Loss: 1.005012 saved
Epoch [10/30] | Train Loss: 0.885687 | Val Loss: 0.975344 saved
Epoch [11/30] | Train Loss: 0.846274 | Val Loss: 0.946589 saved
Epoch [12/30] | Train Loss: 0.804090 | Val Loss: 0.901902 saved
Epoch [13/30] | Train Loss: 0.761203 | Val Loss: 0.854900 saved
Epoch [14/30] | Train Loss: 0.713952 | Val Loss: 0.815066 saved
Epoch [15/30] | Train Loss: 0.668898 | Val Loss: 0.783014 saved
Epoch [16/30] | Train Loss: 0.627111 | Val Loss: 

In [38]:
checkpoint_femba_tuar_tiny_sbpdbp = torch.load(
    best_model_path_femba_tuar_tiny_sbpdbp,
    map_location="cpu"
)

model_femba_tuar_tiny_sbpdbp_best = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
)

model_femba_tuar_tiny_sbpdbp_best.load_state_dict(
    checkpoint_femba_tuar_tiny_sbpdbp["model_state_dict"]
)

model_femba_tuar_tiny_sbpdbp_best = model_femba_tuar_tiny_sbpdbp_best.to(device)
model_femba_tuar_tiny_sbpdbp_best.eval()

print("Loaded best FEMBA TUAR tiny with SBP/DBP loss")
print("Best epoch:", checkpoint_femba_tuar_tiny_sbpdbp["epoch"])
print("Best val loss:", checkpoint_femba_tuar_tiny_sbpdbp["best_val_loss"])

Loaded best FEMBA TUAR tiny with SBP/DBP loss
Best epoch: 29
Best val loss: 0.5024698647090872


In [39]:
y_true_femba_tuar_tiny_sbpdbp, y_pred_femba_tuar_tiny_sbpdbp = evaluate_model_waveform(
    model_femba_tuar_tiny_sbpdbp_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_femba_tuar_tiny_sbpdbp.shape)
print("y_pred:", y_pred_femba_tuar_tiny_sbpdbp.shape)

print("true min/max:", y_true_femba_tuar_tiny_sbpdbp.min(), y_true_femba_tuar_tiny_sbpdbp.max())
print("pred min/max:", y_pred_femba_tuar_tiny_sbpdbp.min(), y_pred_femba_tuar_tiny_sbpdbp.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 41.699753 176.594


In [40]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sbp_true_femba_tuar_tiny_sbpdbp = []
dbp_true_femba_tuar_tiny_sbpdbp = []
sbp_pred_femba_tuar_tiny_sbpdbp = []
dbp_pred_femba_tuar_tiny_sbpdbp = []

for true_sig, pred_sig in zip(y_true_femba_tuar_tiny_sbpdbp, y_pred_femba_tuar_tiny_sbpdbp):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_femba_tuar_tiny_sbpdbp.append(sbp_t)
    dbp_true_femba_tuar_tiny_sbpdbp.append(dbp_t)
    sbp_pred_femba_tuar_tiny_sbpdbp.append(sbp_p)
    dbp_pred_femba_tuar_tiny_sbpdbp.append(dbp_p)

sbp_true_femba_tuar_tiny_sbpdbp = np.array(sbp_true_femba_tuar_tiny_sbpdbp)
dbp_true_femba_tuar_tiny_sbpdbp = np.array(dbp_true_femba_tuar_tiny_sbpdbp)
sbp_pred_femba_tuar_tiny_sbpdbp = np.array(sbp_pred_femba_tuar_tiny_sbpdbp)
dbp_pred_femba_tuar_tiny_sbpdbp = np.array(dbp_pred_femba_tuar_tiny_sbpdbp)

sbp_mae_femba_tuar_tiny_sbpdbp = mean_absolute_error(
    sbp_true_femba_tuar_tiny_sbpdbp,
    sbp_pred_femba_tuar_tiny_sbpdbp
)

dbp_mae_femba_tuar_tiny_sbpdbp = mean_absolute_error(
    dbp_true_femba_tuar_tiny_sbpdbp,
    dbp_pred_femba_tuar_tiny_sbpdbp
)

sbp_rmse_femba_tuar_tiny_sbpdbp = np.sqrt(mean_squared_error(
    sbp_true_femba_tuar_tiny_sbpdbp,
    sbp_pred_femba_tuar_tiny_sbpdbp
))

dbp_rmse_femba_tuar_tiny_sbpdbp = np.sqrt(mean_squared_error(
    dbp_true_femba_tuar_tiny_sbpdbp,
    dbp_pred_femba_tuar_tiny_sbpdbp
))

sbp_r2_femba_tuar_tiny_sbpdbp = r2_score(
    sbp_true_femba_tuar_tiny_sbpdbp,
    sbp_pred_femba_tuar_tiny_sbpdbp
)

dbp_r2_femba_tuar_tiny_sbpdbp = r2_score(
    dbp_true_femba_tuar_tiny_sbpdbp,
    dbp_pred_femba_tuar_tiny_sbpdbp
)

avg_mae_femba_tuar_tiny_sbpdbp = (
    sbp_mae_femba_tuar_tiny_sbpdbp + dbp_mae_femba_tuar_tiny_sbpdbp
) / 2

print("FEMBA TUAR Tiny Full FT with SBP/DBP Loss Results")
print("=================================================")

print("SBP")
print(f"MAE  : {sbp_mae_femba_tuar_tiny_sbpdbp:.3f}")
print(f"RMSE : {sbp_rmse_femba_tuar_tiny_sbpdbp:.3f}")
print(f"R2   : {sbp_r2_femba_tuar_tiny_sbpdbp:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_femba_tuar_tiny_sbpdbp:.3f}")
print(f"RMSE : {dbp_rmse_femba_tuar_tiny_sbpdbp:.3f}")
print(f"R2   : {dbp_r2_femba_tuar_tiny_sbpdbp:.4f}")

print("\nTable format:")
print(f"{sbp_mae_femba_tuar_tiny_sbpdbp:.2f}/{dbp_mae_femba_tuar_tiny_sbpdbp:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_femba_tuar_tiny_sbpdbp:.3f}")

FEMBA TUAR Tiny Full FT with SBP/DBP Loss Results
SBP
MAE  : 5.338
RMSE : 7.571
R2   : 0.7410

DBP
MAE  : 2.871
RMSE : 4.426
R2   : 0.6073

Table format:
5.34/2.87

Average MAE:
4.105


In [41]:
results_femba_tuar_tiny_sbpdbp = pd.DataFrame([{
    "model": "FEMBA",
    "checkpoint": "TUAR tiny",
    "fine_tuning": "full fine-tuning",
    "loss": "Waveform + SBP + DBP loss",
    "seed": SEED,
    "best_epoch": checkpoint_femba_tuar_tiny_sbpdbp["epoch"],
    "best_val_loss": checkpoint_femba_tuar_tiny_sbpdbp["best_val_loss"],
    "sbp_mae": sbp_mae_femba_tuar_tiny_sbpdbp,
    "dbp_mae": dbp_mae_femba_tuar_tiny_sbpdbp,
    "avg_mae": avg_mae_femba_tuar_tiny_sbpdbp,
    "sbp_rmse": sbp_rmse_femba_tuar_tiny_sbpdbp,
    "dbp_rmse": dbp_rmse_femba_tuar_tiny_sbpdbp,
    "sbp_r2": sbp_r2_femba_tuar_tiny_sbpdbp,
    "dbp_r2": dbp_r2_femba_tuar_tiny_sbpdbp
}])

results_femba_tuar_tiny_sbpdbp.to_csv(
    "results_femba_tuar_tiny_full_ft_sbpdbp_loss.csv",
    index=False
)

results_femba_tuar_tiny_sbpdbp

,model,checkpoint,fine_tuning,loss,seed,best_epoch,best_val_loss,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,FEMBA,TUAR tiny,full fine-tuning,Waveform + SBP + DBP loss,42,29,0.50247,5.337622,2.871461,4.104542,7.570586,4.426444,0.740979,0.607318


In [42]:
import gc
import torch

for name in list(globals().keys()):
    if name.startswith("model_femba_tuar_base") or name.startswith("optimizer_femba_tuar_base") or name.startswith("checkpoint_femba_tuar_base"):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Allocated GB: 0.326
Reserved GB: 0.361


In [43]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 2

g = torch.Generator()
g.manual_seed(SEED)

train_loader_base = DataLoader(
    TensorDataset(X_train_scaled, y_train_scaled),
    batch_size=batch_size,
    shuffle=True,
    generator=g
)

val_loader_base = DataLoader(
    TensorDataset(X_val_scaled, y_val_scaled),
    batch_size=batch_size,
    shuffle=False
)

test_loader_base = DataLoader(
    TensorDataset(X_test_scaled, y_test_scaled),
    batch_size=batch_size,
    shuffle=False
)

print("train batches:", len(train_loader_base))
print("val batches:", len(val_loader_base))
print("test batches:", len(test_loader_base))

train batches: 8102
val batches: 2026
test batches: 2532


In [44]:
import os
import glob

search_dirs = [
    "/kaggle/working",
    "/kaggle/input"
]

all_safetensors = []

for d in search_dirs:
    all_safetensors.extend(
        glob.glob(os.path.join(d, "**", "*.safetensors"), recursive=True)
    )

for p in all_safetensors:
    print(p)

/kaggle/working/checkpoints/FEMBA/TUSL/FEMBA_base.safetensors
/kaggle/working/checkpoints/FEMBA/TUSL/FEMBA_large.safetensors
/kaggle/working/checkpoints/FEMBA/TUSL/FEMBA_tiny.safetensors
/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_base.safetensors
/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_large.safetensors
/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_tiny.safetensors
/kaggle/working/checkpoints/FEMBA/TUAB/FEMBA_base.safetensors
/kaggle/working/checkpoints/FEMBA/TUAB/FEMBA_large.safetensors


In [45]:
base_candidates = [
    p for p in all_safetensors
    if "base" in os.path.basename(p).lower()
]

print("Base candidates:")
for p in base_candidates:
    print(p)

assert len(base_candidates) > 0, "No Base checkpoint found. Send me the output from Cell 3."

base_ckpt_path = base_candidates[0]

print("Using:", base_ckpt_path)

Base candidates:
/kaggle/working/checkpoints/FEMBA/TUSL/FEMBA_base.safetensors
/kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_base.safetensors
/kaggle/working/checkpoints/FEMBA/TUAB/FEMBA_base.safetensors
Using: /kaggle/working/checkpoints/FEMBA/TUSL/FEMBA_base.safetensors


In [46]:
from safetensors.torch import load_file

state_base = load_file(base_ckpt_path)

first_key = list(state_base.keys())[0]
print("First key:", first_key)

for k, v in state_base.items():
    if "mamba_blocks.0.mamba_fwd.A_log" in k:
        print(k, v.shape)

for k, v in state_base.items():
    if "patch_embed" in k and len(v.shape) == 4:
        print("patch_embed:", k, v.shape)

First key: decoder.conv_transpose.bias
mamba_blocks.0.mamba_fwd.A_log torch.Size([1540, 16])
patch_embed: patch_embed.proj.weight torch.Size([35, 1, 2, 16])


In [47]:
import re

a_log_key = None

for k in state_base.keys():
    if "mamba_blocks.0.mamba_fwd.A_log" in k:
        a_log_key = k
        break

assert a_log_key is not None, "Could not find A_log key"

inner_dim = state_base[a_log_key].shape[0]
embed_dim_base = inner_dim // 4

block_ids = []

for k in state_base.keys():
    m = re.search(r"mamba_blocks\.(\d+)", k)
    if m:
        block_ids.append(int(m.group(1)))

num_blocks_base = max(block_ids) + 1

print("inner_dim:", inner_dim)
print("embed_dim_base:", embed_dim_base)
print("num_blocks_base:", num_blocks_base)

inner_dim: 1540
embed_dim_base: 385
num_blocks_base: 12


In [49]:
tuar_base_candidates = [
    p for p in all_safetensors
    if "tuar" in p.lower() and "base" in os.path.basename(p).lower()
]

print("TUAR Base candidates:")
for i, p in enumerate(tuar_base_candidates):
    print(i, p)

assert len(tuar_base_candidates) > 0, "No TUAR Base checkpoint found"

base_ckpt_path = tuar_base_candidates[0]

print("Using TUAR Base:", base_ckpt_path)

TUAR Base candidates:
0 /kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_base.safetensors
Using TUAR Base: /kaggle/working/checkpoints/FEMBA/TUAR/FEMBA_base.safetensors


In [50]:
from safetensors.torch import load_file
import re

state_base = load_file(base_ckpt_path)

a_log_key = None

for k in state_base.keys():
    if "mamba_blocks.0.mamba_fwd.A_log" in k:
        a_log_key = k
        break

assert a_log_key is not None, "Could not find A_log key"

inner_dim = state_base[a_log_key].shape[0]
embed_dim_base = inner_dim // 4

block_ids = []

for k in state_base.keys():
    m = re.search(r"mamba_blocks\.(\d+)", k)
    if m:
        block_ids.append(int(m.group(1)))

num_blocks_base = max(block_ids) + 1

print("A_log:", a_log_key, state_base[a_log_key].shape)
print("inner_dim:", inner_dim)
print("embed_dim_base:", embed_dim_base)
print("num_blocks_base:", num_blocks_base)

A_log: mamba_blocks.0.mamba_fwd.A_log torch.Size([1540, 16])
inner_dim: 1540
embed_dim_base: 385
num_blocks_base: 12


In [51]:
import gc
import torch

for name in list(globals().keys()):
    if (
        name.startswith("model_femba")
        or name.startswith("optimizer_femba")
        or name.startswith("checkpoint_femba")
        or name.startswith("matched_state")
    ):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Allocated GB: 14.093
Reserved GB: 14.357


In [52]:
from torch.utils.data import TensorDataset, DataLoader

batch_size_base = 1

g = torch.Generator()
g.manual_seed(SEED)

train_loader_base = DataLoader(
    TensorDataset(X_train_scaled, y_train_scaled),
    batch_size=batch_size_base,
    shuffle=True,
    generator=g
)

val_loader_base = DataLoader(
    TensorDataset(X_val_scaled, y_val_scaled),
    batch_size=batch_size_base,
    shuffle=False
)

test_loader_base = DataLoader(
    TensorDataset(X_test_scaled, y_test_scaled),
    batch_size=batch_size_base,
    shuffle=False
)

print("train batches:", len(train_loader_base))
print("val batches:", len(val_loader_base))
print("test batches:", len(test_loader_base))

train batches: 16204
val batches: 4051
test batches: 5064


In [53]:
gc.collect()
torch.cuda.empty_cache()

model_femba_tuar_base_sbpdbp = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim_base,
    num_blocks=num_blocks_base
).to(device)

print("FEMBA TUAR Base model created")

OutOfMemoryError: CUDA out of memory. Tried to allocate 548.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 67.81 MiB is free. Including non-PyTorch memory, this process has 14.49 GiB memory in use. Of the allocated memory 14.09 GiB is allocated by PyTorch, and 270.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# two stage on tiny

In [54]:
import gc
import torch

for name in list(globals().keys()):
    if (
        name.startswith("model_femba_tuar_tiny_twostage")
        or name.startswith("optimizer_femba_tuar_tiny_twostage")
        or name.startswith("checkpoint_femba_tuar_tiny_twostage")
    ):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Allocated GB: 0.049
Reserved GB: 0.1


In [55]:
model_femba_tuar_tiny_twostage = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
).to(device)

model_dict = model_femba_tuar_tiny_twostage.state_dict()
matched_state_twostage = {}

for ckpt_key, ckpt_value in state.items():
    candidates = [
        ckpt_key,
        "femba." + ckpt_key
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == ckpt_value.shape:
            matched_state_twostage[candidate] = ckpt_value
            break

missing, unexpected = model_femba_tuar_tiny_twostage.load_state_dict(
    matched_state_twostage,
    strict=False
)

print("Loaded tensors:", len(matched_state_twostage))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Loaded tensors: 42
Missing keys: 18
Unexpected keys: 0


In [56]:
for param in model_femba_tuar_tiny_twostage.femba.parameters():
    param.requires_grad = False

for param in model_femba_tuar_tiny_twostage.regression_head.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_femba_tuar_tiny_twostage.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_femba_tuar_tiny_twostage.parameters()
)

print("Trainable params:", trainable_params)
print("Total params:", total_params)

Trainable params: 259441
Total params: 8575584


In [57]:
criterion_femba_tuar_tiny_twostage = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_femba_tuar_tiny_twostage_stage1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_femba_tuar_tiny_twostage.parameters()),
    lr=1e-3,
    weight_decay=1e-2
)

num_epochs_stage1 = 10
history_femba_tuar_tiny_twostage = []

for epoch in range(num_epochs_stage1):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny_twostage,
        train_loader,
        optimizer_femba_tuar_tiny_twostage_stage1,
        criterion_femba_tuar_tiny_twostage,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny_twostage,
        val_loader,
        criterion_femba_tuar_tiny_twostage,
        device
    )

    history_femba_tuar_tiny_twostage.append({
        "method": "FEMBA_TUAR_tiny_two_stage",
        "stage": 1,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 1e-3,
        "seed": SEED
    })

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

Stage 1 Epoch [1/10] | Train Loss: 1.400421 | Val Loss: 1.417228
Stage 1 Epoch [2/10] | Train Loss: 1.387108 | Val Loss: 1.414366
Stage 1 Epoch [3/10] | Train Loss: 1.386258 | Val Loss: 1.414007
Stage 1 Epoch [4/10] | Train Loss: 1.382095 | Val Loss: 1.419319
Stage 1 Epoch [5/10] | Train Loss: 1.380273 | Val Loss: 1.420403
Stage 1 Epoch [6/10] | Train Loss: 1.378575 | Val Loss: 1.405653
Stage 1 Epoch [7/10] | Train Loss: 1.375648 | Val Loss: 1.410906
Stage 1 Epoch [8/10] | Train Loss: 1.371844 | Val Loss: 1.407669
Stage 1 Epoch [9/10] | Train Loss: 1.369962 | Val Loss: 1.402043
Stage 1 Epoch [10/10] | Train Loss: 1.365318 | Val Loss: 1.400029


In [58]:
for param in model_femba_tuar_tiny_twostage.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_femba_tuar_tiny_twostage.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_femba_tuar_tiny_twostage.parameters()
)

print("Trainable params:", trainable_params)
print("Total params:", total_params)

Trainable params: 8575584
Total params: 8575584


In [59]:
optimizer_femba_tuar_tiny_twostage_stage2 = torch.optim.AdamW(
    model_femba_tuar_tiny_twostage.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

num_epochs_stage2 = 20
best_val_loss_femba_tuar_tiny_twostage = float("inf")
best_model_path_femba_tuar_tiny_twostage = "femba_tuar_tiny_two_stage_sbpdbp_loss_seed42.pth"

for epoch in range(num_epochs_stage2):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny_twostage,
        train_loader,
        optimizer_femba_tuar_tiny_twostage_stage2,
        criterion_femba_tuar_tiny_twostage,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny_twostage,
        val_loader,
        criterion_femba_tuar_tiny_twostage,
        device
    )

    history_femba_tuar_tiny_twostage.append({
        "method": "FEMBA_TUAR_tiny_two_stage",
        "stage": 2,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 1e-4,
        "seed": SEED
    })

    if val_loss < best_val_loss_femba_tuar_tiny_twostage:
        best_val_loss_femba_tuar_tiny_twostage = val_loss

        torch.save({
            "method": "FEMBA_TUAR_tiny_two_stage_sbpdbp_loss",
            "model_state_dict": model_femba_tuar_tiny_twostage.state_dict(),
            "optimizer_state_dict": optimizer_femba_tuar_tiny_twostage_stage2.state_dict(),
            "best_val_loss": float(best_val_loss_femba_tuar_tiny_twostage),
            "stage1_epochs": num_epochs_stage1,
            "stage2_epoch": epoch + 1,
            "lr_stage1": 1e-3,
            "lr_stage2": 1e-4,
            "seed": SEED
        }, best_model_path_femba_tuar_tiny_twostage)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Stage 2 Epoch [1/20] | Train Loss: 1.256008 | Val Loss: 1.174239 saved
Stage 2 Epoch [2/20] | Train Loss: 1.099621 | Val Loss: 1.074373 saved
Stage 2 Epoch [3/20] | Train Loss: 1.078473 | Val Loss: 1.277207 
Stage 2 Epoch [4/20] | Train Loss: 1.067019 | Val Loss: 1.047613 saved
Stage 2 Epoch [5/20] | Train Loss: 0.992078 | Val Loss: 1.004098 saved
Stage 2 Epoch [6/20] | Train Loss: 0.973560 | Val Loss: 0.998637 saved
Stage 2 Epoch [7/20] | Train Loss: 0.961017 | Val Loss: 0.974058 saved
Stage 2 Epoch [8/20] | Train Loss: 0.910758 | Val Loss: 0.940770 saved
Stage 2 Epoch [9/20] | Train Loss: 0.887967 | Val Loss: 0.928404 saved
Stage 2 Epoch [10/20] | Train Loss: 0.864462 | Val Loss: 0.914463 saved
Stage 2 Epoch [11/20] | Train Loss: 0.852870 | Val Loss: 0.879253 saved
Stage 2 Epoch [12/20] | Train Loss: 0.802559 | Val Loss: 0.867306 saved
Stage 2 Epoch [13/20] | Train Loss: 0.799302 | Val Loss: 0.835332 saved
Stage 2 Epoch [14/20] | Train Loss: 0.782933 | Val Loss: 0.810485 saved
Stage 

In [60]:
checkpoint_femba_tuar_tiny_twostage = torch.load(
    best_model_path_femba_tuar_tiny_twostage,
    map_location="cpu"
)

model_femba_tuar_tiny_twostage_best = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
)

model_femba_tuar_tiny_twostage_best.load_state_dict(
    checkpoint_femba_tuar_tiny_twostage["model_state_dict"]
)

model_femba_tuar_tiny_twostage_best = model_femba_tuar_tiny_twostage_best.to(device)
model_femba_tuar_tiny_twostage_best.eval()

print("Loaded best FEMBA TUAR Tiny Two-Stage model")
print("Best Stage 2 epoch:", checkpoint_femba_tuar_tiny_twostage["stage2_epoch"])
print("Best val loss:", checkpoint_femba_tuar_tiny_twostage["best_val_loss"])

Loaded best FEMBA TUAR Tiny Two-Stage model
Best Stage 2 epoch: 20
Best val loss: 0.7310654834534498


In [61]:
y_true_femba_tuar_tiny_twostage, y_pred_femba_tuar_tiny_twostage = evaluate_model_waveform(
    model_femba_tuar_tiny_twostage_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_femba_tuar_tiny_twostage.shape)
print("y_pred:", y_pred_femba_tuar_tiny_twostage.shape)

print("true min/max:", y_true_femba_tuar_tiny_twostage.min(), y_true_femba_tuar_tiny_twostage.max())
print("pred min/max:", y_pred_femba_tuar_tiny_twostage.min(), y_pred_femba_tuar_tiny_twostage.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 55.10406 175.00418


In [62]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sbp_true_femba_tuar_tiny_twostage = []
dbp_true_femba_tuar_tiny_twostage = []
sbp_pred_femba_tuar_tiny_twostage = []
dbp_pred_femba_tuar_tiny_twostage = []

for true_sig, pred_sig in zip(y_true_femba_tuar_tiny_twostage, y_pred_femba_tuar_tiny_twostage):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_femba_tuar_tiny_twostage.append(sbp_t)
    dbp_true_femba_tuar_tiny_twostage.append(dbp_t)
    sbp_pred_femba_tuar_tiny_twostage.append(sbp_p)
    dbp_pred_femba_tuar_tiny_twostage.append(dbp_p)

sbp_true_femba_tuar_tiny_twostage = np.array(sbp_true_femba_tuar_tiny_twostage)
dbp_true_femba_tuar_tiny_twostage = np.array(dbp_true_femba_tuar_tiny_twostage)
sbp_pred_femba_tuar_tiny_twostage = np.array(sbp_pred_femba_tuar_tiny_twostage)
dbp_pred_femba_tuar_tiny_twostage = np.array(dbp_pred_femba_tuar_tiny_twostage)

sbp_mae_femba_tuar_tiny_twostage = mean_absolute_error(
    sbp_true_femba_tuar_tiny_twostage,
    sbp_pred_femba_tuar_tiny_twostage
)

dbp_mae_femba_tuar_tiny_twostage = mean_absolute_error(
    dbp_true_femba_tuar_tiny_twostage,
    dbp_pred_femba_tuar_tiny_twostage
)

sbp_rmse_femba_tuar_tiny_twostage = np.sqrt(mean_squared_error(
    sbp_true_femba_tuar_tiny_twostage,
    sbp_pred_femba_tuar_tiny_twostage
))

dbp_rmse_femba_tuar_tiny_twostage = np.sqrt(mean_squared_error(
    dbp_true_femba_tuar_tiny_twostage,
    dbp_pred_femba_tuar_tiny_twostage
))

sbp_r2_femba_tuar_tiny_twostage = r2_score(
    sbp_true_femba_tuar_tiny_twostage,
    sbp_pred_femba_tuar_tiny_twostage
)

dbp_r2_femba_tuar_tiny_twostage = r2_score(
    dbp_true_femba_tuar_tiny_twostage,
    dbp_pred_femba_tuar_tiny_twostage
)

avg_mae_femba_tuar_tiny_twostage = (
    sbp_mae_femba_tuar_tiny_twostage + dbp_mae_femba_tuar_tiny_twostage
) / 2

print("FEMBA TUAR Tiny Two-Stage Full FT with SBP/DBP Loss Results")
print("===========================================================")

print("SBP")
print(f"MAE  : {sbp_mae_femba_tuar_tiny_twostage:.3f}")
print(f"RMSE : {sbp_rmse_femba_tuar_tiny_twostage:.3f}")
print(f"R2   : {sbp_r2_femba_tuar_tiny_twostage:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_femba_tuar_tiny_twostage:.3f}")
print(f"RMSE : {dbp_rmse_femba_tuar_tiny_twostage:.3f}")
print(f"R2   : {dbp_r2_femba_tuar_tiny_twostage:.4f}")

print("\nTable format:")
print(f"{sbp_mae_femba_tuar_tiny_twostage:.2f}/{dbp_mae_femba_tuar_tiny_twostage:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_femba_tuar_tiny_twostage:.3f}")

FEMBA TUAR Tiny Two-Stage Full FT with SBP/DBP Loss Results
SBP
MAE  : 5.638
RMSE : 7.963
R2   : 0.7134

DBP
MAE  : 2.949
RMSE : 4.451
R2   : 0.6030

Table format:
5.64/2.95

Average MAE:
4.293


In [63]:
results_femba_tuar_tiny_twostage = pd.DataFrame([{
    "model": "FEMBA",
    "checkpoint": "TUAR tiny",
    "fine_tuning": "two-stage full fine-tuning",
    "loss": "Waveform + SBP + DBP loss",
    "seed": SEED,
    "stage1_epochs": checkpoint_femba_tuar_tiny_twostage["stage1_epochs"],
    "best_stage2_epoch": checkpoint_femba_tuar_tiny_twostage["stage2_epoch"],
    "best_val_loss": checkpoint_femba_tuar_tiny_twostage["best_val_loss"],
    "stage1_head_lr": checkpoint_femba_tuar_tiny_twostage["lr_stage1"],
    "stage2_backbone_lr": 2e-5,
    "stage2_head_lr": 1e-4,
    "sbp_mae": sbp_mae_femba_tuar_tiny_twostage,
    "dbp_mae": dbp_mae_femba_tuar_tiny_twostage,
    "avg_mae": avg_mae_femba_tuar_tiny_twostage,
    "sbp_rmse": sbp_rmse_femba_tuar_tiny_twostage,
    "dbp_rmse": dbp_rmse_femba_tuar_tiny_twostage,
    "sbp_r2": sbp_r2_femba_tuar_tiny_twostage,
    "dbp_r2": dbp_r2_femba_tuar_tiny_twostage
}])

results_femba_tuar_tiny_twostage.to_csv(
    "results_femba_tuar_tiny_two_stage_sbpdbp_loss.csv",
    index=False
)

results_femba_tuar_tiny_twostage

,model,checkpoint,fine_tuning,loss,seed,stage1_epochs,best_stage2_epoch,best_val_loss,stage1_head_lr,stage2_backbone_lr,stage2_head_lr,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,FEMBA,TUAR tiny,two-stage full fine-tuning,Waveform + SBP + DBP loss,42,10,20,0.731065,0.001,0.00002,0.0001,5.637656,2.94899,4.293323,7.963416,4.450988,0.713401,0.602951


# Tw stage Head 1e-4 10ep 
Full FT Backbone 3e-5 Head 1e-4


In [64]:
import gc
import torch

for name in list(globals().keys()):
    if (
        name.startswith("model_femba_tuar_tiny_bestsetting")
        or name.startswith("optimizer_femba_tuar_tiny_bestsetting")
        or name.startswith("checkpoint_femba_tuar_tiny_bestsetting")
    ):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Allocated GB: 0.174
Reserved GB: 0.205


In [65]:
model_femba_tuar_tiny_bestsetting = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
).to(device)

model_dict = model_femba_tuar_tiny_bestsetting.state_dict()
matched_state_bestsetting = {}

for ckpt_key, ckpt_value in state.items():
    candidates = [
        ckpt_key,
        "femba." + ckpt_key
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == ckpt_value.shape:
            matched_state_bestsetting[candidate] = ckpt_value
            break

missing, unexpected = model_femba_tuar_tiny_bestsetting.load_state_dict(
    matched_state_bestsetting,
    strict=False
)

print("Loaded tensors:", len(matched_state_bestsetting))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Loaded tensors: 42
Missing keys: 18
Unexpected keys: 0


In [66]:
for param in model_femba_tuar_tiny_bestsetting.femba.parameters():
    param.requires_grad = False

for param in model_femba_tuar_tiny_bestsetting.regression_head.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_femba_tuar_tiny_bestsetting.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_femba_tuar_tiny_bestsetting.parameters()
)

print("Trainable params:", trainable_params)
print("Total params:", total_params)

Trainable params: 259441
Total params: 8575584


In [67]:
criterion_femba_tuar_tiny_bestsetting = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_femba_tuar_tiny_bestsetting_stage1 = torch.optim.AdamW(
    model_femba_tuar_tiny_bestsetting.regression_head.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

num_epochs_stage1_bestsetting = 10
history_femba_tuar_tiny_bestsetting = []

for epoch in range(num_epochs_stage1_bestsetting):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny_bestsetting,
        train_loader,
        optimizer_femba_tuar_tiny_bestsetting_stage1,
        criterion_femba_tuar_tiny_bestsetting,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny_bestsetting,
        val_loader,
        criterion_femba_tuar_tiny_bestsetting,
        device
    )

    history_femba_tuar_tiny_bestsetting.append({
        "method": "FEMBA_TUAR_tiny_two_stage_bestsetting",
        "stage": 1,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "head_lr": 1e-4,
        "backbone_lr": 0.0,
        "seed": SEED
    })

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1_bestsetting}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

Stage 1 Epoch [1/10] | Train Loss: 1.427476 | Val Loss: 1.420772
Stage 1 Epoch [2/10] | Train Loss: 1.384025 | Val Loss: 1.417409
Stage 1 Epoch [3/10] | Train Loss: 1.382105 | Val Loss: 1.412894
Stage 1 Epoch [4/10] | Train Loss: 1.379813 | Val Loss: 1.411865
Stage 1 Epoch [5/10] | Train Loss: 1.377961 | Val Loss: 1.412005
Stage 1 Epoch [6/10] | Train Loss: 1.375994 | Val Loss: 1.409743
Stage 1 Epoch [7/10] | Train Loss: 1.374835 | Val Loss: 1.409774
Stage 1 Epoch [8/10] | Train Loss: 1.373935 | Val Loss: 1.415814
Stage 1 Epoch [9/10] | Train Loss: 1.372703 | Val Loss: 1.405554
Stage 1 Epoch [10/10] | Train Loss: 1.370866 | Val Loss: 1.404100


In [68]:
for param in model_femba_tuar_tiny_bestsetting.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_femba_tuar_tiny_bestsetting.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_femba_tuar_tiny_bestsetting.parameters()
)

print("Trainable params:", trainable_params)
print("Total params:", total_params)

Trainable params: 8575584
Total params: 8575584


In [69]:
optimizer_femba_tuar_tiny_bestsetting_stage2 = torch.optim.AdamW(
    [
        {
            "params": model_femba_tuar_tiny_bestsetting.femba.parameters(),
            "lr": 3e-5
        },
        {
            "params": model_femba_tuar_tiny_bestsetting.regression_head.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-2
)

num_epochs_stage2_bestsetting = 20
best_val_loss_femba_tuar_tiny_bestsetting = float("inf")
best_model_path_femba_tuar_tiny_bestsetting = "femba_tuar_tiny_two_stage_head1e4_backbone3e5_sbpdbp_seed42.pth"

for epoch in range(num_epochs_stage2_bestsetting):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny_bestsetting,
        train_loader,
        optimizer_femba_tuar_tiny_bestsetting_stage2,
        criterion_femba_tuar_tiny_bestsetting,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny_bestsetting,
        val_loader,
        criterion_femba_tuar_tiny_bestsetting,
        device
    )

    history_femba_tuar_tiny_bestsetting.append({
        "method": "FEMBA_TUAR_tiny_two_stage_bestsetting",
        "stage": 2,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "head_lr": 1e-4,
        "backbone_lr": 3e-5,
        "seed": SEED
    })

    if val_loss < best_val_loss_femba_tuar_tiny_bestsetting:
        best_val_loss_femba_tuar_tiny_bestsetting = val_loss

        torch.save({
            "method": "FEMBA_TUAR_tiny_two_stage_head1e4_backbone3e5_sbpdbp_loss",
            "model_state_dict": model_femba_tuar_tiny_bestsetting.state_dict(),
            "optimizer_state_dict": optimizer_femba_tuar_tiny_bestsetting_stage2.state_dict(),
            "best_val_loss": float(best_val_loss_femba_tuar_tiny_bestsetting),
            "stage1_epochs": num_epochs_stage1_bestsetting,
            "stage2_epoch": epoch + 1,
            "lr_stage1_head": 1e-4,
            "lr_stage2_backbone": 3e-5,
            "lr_stage2_head": 1e-4,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_femba_tuar_tiny_bestsetting)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2_bestsetting}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Stage 2 Epoch [1/20] | Train Loss: 1.271811 | Val Loss: 1.222829 saved
Stage 2 Epoch [2/20] | Train Loss: 1.147243 | Val Loss: 1.129679 saved
Stage 2 Epoch [3/20] | Train Loss: 1.082221 | Val Loss: 1.108122 saved
Stage 2 Epoch [4/20] | Train Loss: 1.044472 | Val Loss: 1.077341 saved
Stage 2 Epoch [5/20] | Train Loss: 1.010572 | Val Loss: 1.081226 
Stage 2 Epoch [6/20] | Train Loss: 0.984182 | Val Loss: 1.046289 saved
Stage 2 Epoch [7/20] | Train Loss: 0.963577 | Val Loss: 1.040608 saved
Stage 2 Epoch [8/20] | Train Loss: 0.951206 | Val Loss: 1.042870 
Stage 2 Epoch [9/20] | Train Loss: 0.929344 | Val Loss: 1.039289 saved
Stage 2 Epoch [10/20] | Train Loss: 0.916929 | Val Loss: 1.024526 saved
Stage 2 Epoch [11/20] | Train Loss: 0.905203 | Val Loss: 1.022851 saved
Stage 2 Epoch [12/20] | Train Loss: 0.896912 | Val Loss: 1.029689 
Stage 2 Epoch [13/20] | Train Loss: 0.888673 | Val Loss: 1.024186 
Stage 2 Epoch [14/20] | Train Loss: 0.873152 | Val Loss: 1.000448 saved
Stage 2 Epoch [15/20]

In [70]:
checkpoint_femba_tuar_tiny_bestsetting = torch.load(
    best_model_path_femba_tuar_tiny_bestsetting,
    map_location="cpu"
)

model_femba_tuar_tiny_bestsetting_best = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
)

model_femba_tuar_tiny_bestsetting_best.load_state_dict(
    checkpoint_femba_tuar_tiny_bestsetting["model_state_dict"]
)

model_femba_tuar_tiny_bestsetting_best = model_femba_tuar_tiny_bestsetting_best.to(device)
model_femba_tuar_tiny_bestsetting_best.eval()

print("Loaded best FEMBA TUAR Tiny best-setting two-stage model")
print("Best Stage 2 epoch:", checkpoint_femba_tuar_tiny_bestsetting["stage2_epoch"])
print("Best val loss:", checkpoint_femba_tuar_tiny_bestsetting["best_val_loss"])
print("Stage 1 Head LR:", checkpoint_femba_tuar_tiny_bestsetting["lr_stage1_head"])
print("Stage 2 Backbone LR:", checkpoint_femba_tuar_tiny_bestsetting["lr_stage2_backbone"])
print("Stage 2 Head LR:", checkpoint_femba_tuar_tiny_bestsetting["lr_stage2_head"])

Loaded best FEMBA TUAR Tiny best-setting two-stage model
Best Stage 2 epoch: 20
Best val loss: 0.8257770584236396
Stage 1 Head LR: 0.0001
Stage 2 Backbone LR: 3e-05
Stage 2 Head LR: 0.0001


In [71]:
y_true_femba_tuar_tiny_bestsetting, y_pred_femba_tuar_tiny_bestsetting = evaluate_model_waveform(
    model_femba_tuar_tiny_bestsetting_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_femba_tuar_tiny_bestsetting.shape)
print("y_pred:", y_pred_femba_tuar_tiny_bestsetting.shape)

print("true min/max:", y_true_femba_tuar_tiny_bestsetting.min(), y_true_femba_tuar_tiny_bestsetting.max())
print("pred min/max:", y_pred_femba_tuar_tiny_bestsetting.min(), y_pred_femba_tuar_tiny_bestsetting.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 52.443058 189.88905


In [72]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sbp_true_femba_tuar_tiny_bestsetting = []
dbp_true_femba_tuar_tiny_bestsetting = []
sbp_pred_femba_tuar_tiny_bestsetting = []
dbp_pred_femba_tuar_tiny_bestsetting = []

for true_sig, pred_sig in zip(y_true_femba_tuar_tiny_bestsetting, y_pred_femba_tuar_tiny_bestsetting):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_femba_tuar_tiny_bestsetting.append(sbp_t)
    dbp_true_femba_tuar_tiny_bestsetting.append(dbp_t)
    sbp_pred_femba_tuar_tiny_bestsetting.append(sbp_p)
    dbp_pred_femba_tuar_tiny_bestsetting.append(dbp_p)

sbp_true_femba_tuar_tiny_bestsetting = np.array(sbp_true_femba_tuar_tiny_bestsetting)
dbp_true_femba_tuar_tiny_bestsetting = np.array(dbp_true_femba_tuar_tiny_bestsetting)
sbp_pred_femba_tuar_tiny_bestsetting = np.array(sbp_pred_femba_tuar_tiny_bestsetting)
dbp_pred_femba_tuar_tiny_bestsetting = np.array(dbp_pred_femba_tuar_tiny_bestsetting)

sbp_mae_femba_tuar_tiny_bestsetting = mean_absolute_error(
    sbp_true_femba_tuar_tiny_bestsetting,
    sbp_pred_femba_tuar_tiny_bestsetting
)

dbp_mae_femba_tuar_tiny_bestsetting = mean_absolute_error(
    dbp_true_femba_tuar_tiny_bestsetting,
    dbp_pred_femba_tuar_tiny_bestsetting
)

sbp_rmse_femba_tuar_tiny_bestsetting = np.sqrt(mean_squared_error(
    sbp_true_femba_tuar_tiny_bestsetting,
    sbp_pred_femba_tuar_tiny_bestsetting
))

dbp_rmse_femba_tuar_tiny_bestsetting = np.sqrt(mean_squared_error(
    dbp_true_femba_tuar_tiny_bestsetting,
    dbp_pred_femba_tuar_tiny_bestsetting
))

sbp_r2_femba_tuar_tiny_bestsetting = r2_score(
    sbp_true_femba_tuar_tiny_bestsetting,
    sbp_pred_femba_tuar_tiny_bestsetting
)

dbp_r2_femba_tuar_tiny_bestsetting = r2_score(
    dbp_true_femba_tuar_tiny_bestsetting,
    dbp_pred_femba_tuar_tiny_bestsetting
)

avg_mae_femba_tuar_tiny_bestsetting = (
    sbp_mae_femba_tuar_tiny_bestsetting + dbp_mae_femba_tuar_tiny_bestsetting
) / 2

print("FEMBA TUAR Tiny Two-Stage Best Setting Results")
print("==============================================")
print("Setting: Stage 1 Head 1e-4 10 epochs; Stage 2 Backbone 3e-5 Head 1e-4")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_femba_tuar_tiny_bestsetting:.3f}")
print(f"RMSE : {sbp_rmse_femba_tuar_tiny_bestsetting:.3f}")
print(f"R2   : {sbp_r2_femba_tuar_tiny_bestsetting:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_femba_tuar_tiny_bestsetting:.3f}")
print(f"RMSE : {dbp_rmse_femba_tuar_tiny_bestsetting:.3f}")
print(f"R2   : {dbp_r2_femba_tuar_tiny_bestsetting:.4f}")

print("\nTable format:")
print(f"{sbp_mae_femba_tuar_tiny_bestsetting:.2f}/{dbp_mae_femba_tuar_tiny_bestsetting:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_femba_tuar_tiny_bestsetting:.3f}")

FEMBA TUAR Tiny Two-Stage Best Setting Results
Setting: Stage 1 Head 1e-4 10 epochs; Stage 2 Backbone 3e-5 Head 1e-4
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 5.945
RMSE : 8.293
R2   : 0.6892

DBP
MAE  : 2.955
RMSE : 4.530
R2   : 0.5888

Table format:
5.94/2.95

Average MAE:
4.450


In [73]:
results_femba_tuar_tiny_bestsetting = pd.DataFrame([{
    "model": "FEMBA",
    "checkpoint": "TUAR tiny",
    "fine_tuning": "two-stage full fine-tuning",
    "loss": "Waveform + SBP + DBP loss",
    "stage1": "head only",
    "stage1_epochs": checkpoint_femba_tuar_tiny_bestsetting["stage1_epochs"],
    "stage1_head_lr": checkpoint_femba_tuar_tiny_bestsetting["lr_stage1_head"],
    "stage2": "full fine-tuning",
    "best_stage2_epoch": checkpoint_femba_tuar_tiny_bestsetting["stage2_epoch"],
    "stage2_backbone_lr": checkpoint_femba_tuar_tiny_bestsetting["lr_stage2_backbone"],
    "stage2_head_lr": checkpoint_femba_tuar_tiny_bestsetting["lr_stage2_head"],
    "seed": SEED,
    "best_val_loss": checkpoint_femba_tuar_tiny_bestsetting["best_val_loss"],
    "sbp_mae": sbp_mae_femba_tuar_tiny_bestsetting,
    "dbp_mae": dbp_mae_femba_tuar_tiny_bestsetting,
    "avg_mae": avg_mae_femba_tuar_tiny_bestsetting,
    "sbp_rmse": sbp_rmse_femba_tuar_tiny_bestsetting,
    "dbp_rmse": dbp_rmse_femba_tuar_tiny_bestsetting,
    "sbp_r2": sbp_r2_femba_tuar_tiny_bestsetting,
    "dbp_r2": dbp_r2_femba_tuar_tiny_bestsetting
}])

results_femba_tuar_tiny_bestsetting.to_csv(
    "results_femba_tuar_tiny_two_stage_head1e4_backbone3e5_sbpdbp_loss.csv",
    index=False
)

results_femba_tuar_tiny_bestsetting

,model,checkpoint,fine_tuning,loss,stage1,stage1_epochs,stage1_head_lr,stage2,best_stage2_epoch,stage2_backbone_lr,stage2_head_lr,seed,best_val_loss,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,FEMBA,TUAR tiny,two-stage full fine-tuning,Waveform + SBP + DBP loss,head only,10,0.0001,full fine-tuning,20,0.00003,0.0001,42,0.825777,5.944732,2.954798,4.449765,8.293478,4.529557,0.689151,0.58881


In [74]:
import gc
import torch

for name in list(globals().keys()):
    if (
        name.startswith("model_femba_tuar_tiny_2e5")
        or name.startswith("optimizer_femba_tuar_tiny_2e5")
        or name.startswith("checkpoint_femba_tuar_tiny_2e5")
    ):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Allocated GB: 0.331
Reserved GB: 0.365


In [75]:
model_femba_tuar_tiny_2e5 = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
).to(device)

model_dict = model_femba_tuar_tiny_2e5.state_dict()
matched_state_2e5 = {}

for ckpt_key, ckpt_value in state.items():
    candidates = [
        ckpt_key,
        "femba." + ckpt_key
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == ckpt_value.shape:
            matched_state_2e5[candidate] = ckpt_value
            break

missing, unexpected = model_femba_tuar_tiny_2e5.load_state_dict(
    matched_state_2e5,
    strict=False
)

print("Loaded tensors:", len(matched_state_2e5))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Loaded tensors: 42
Missing keys: 18
Unexpected keys: 0


In [76]:
for param in model_femba_tuar_tiny_2e5.femba.parameters():
    param.requires_grad = False

for param in model_femba_tuar_tiny_2e5.regression_head.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_femba_tuar_tiny_2e5.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_femba_tuar_tiny_2e5.parameters()
)

print("Trainable params:", trainable_params)
print("Total params:", total_params)

Trainable params: 259441
Total params: 8575584


In [77]:
criterion_femba_tuar_tiny_2e5 = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_femba_tuar_tiny_2e5_stage1 = torch.optim.AdamW(
    model_femba_tuar_tiny_2e5.regression_head.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

num_epochs_stage1_2e5 = 10
history_femba_tuar_tiny_2e5 = []

for epoch in range(num_epochs_stage1_2e5):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny_2e5,
        train_loader,
        optimizer_femba_tuar_tiny_2e5_stage1,
        criterion_femba_tuar_tiny_2e5,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny_2e5,
        val_loader,
        criterion_femba_tuar_tiny_2e5,
        device
    )

    history_femba_tuar_tiny_2e5.append({
        "method": "FEMBA_TUAR_tiny_two_stage_2e5",
        "stage": 1,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "head_lr": 1e-4,
        "backbone_lr": 0.0,
        "seed": SEED
    })

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1_2e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

Stage 1 Epoch [1/10] | Train Loss: 1.430550 | Val Loss: 1.416989
Stage 1 Epoch [2/10] | Train Loss: 1.384323 | Val Loss: 1.415011
Stage 1 Epoch [3/10] | Train Loss: 1.382295 | Val Loss: 1.419336
Stage 1 Epoch [4/10] | Train Loss: 1.379641 | Val Loss: 1.412077
Stage 1 Epoch [5/10] | Train Loss: 1.377598 | Val Loss: 1.410090
Stage 1 Epoch [6/10] | Train Loss: 1.377054 | Val Loss: 1.418035
Stage 1 Epoch [7/10] | Train Loss: 1.375220 | Val Loss: 1.410645
Stage 1 Epoch [8/10] | Train Loss: 1.374234 | Val Loss: 1.406602
Stage 1 Epoch [9/10] | Train Loss: 1.372455 | Val Loss: 1.406640
Stage 1 Epoch [10/10] | Train Loss: 1.371331 | Val Loss: 1.406116


In [78]:
for param in model_femba_tuar_tiny_2e5.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_femba_tuar_tiny_2e5.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_femba_tuar_tiny_2e5.parameters()
)

print("Trainable params:", trainable_params)
print("Total params:", total_params)

Trainable params: 8575584
Total params: 8575584


In [79]:
optimizer_femba_tuar_tiny_2e5_stage2 = torch.optim.AdamW(
    [
        {
            "params": model_femba_tuar_tiny_2e5.femba.parameters(),
            "lr": 2e-5
        },
        {
            "params": model_femba_tuar_tiny_2e5.regression_head.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-2
)

num_epochs_stage2_2e5 = 20
best_val_loss_femba_tuar_tiny_2e5 = float("inf")
best_model_path_femba_tuar_tiny_2e5 = "femba_tuar_tiny_two_stage_head1e4_backbone2e5_sbpdbp_seed42.pth"

for epoch in range(num_epochs_stage2_2e5):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny_2e5,
        train_loader,
        optimizer_femba_tuar_tiny_2e5_stage2,
        criterion_femba_tuar_tiny_2e5,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny_2e5,
        val_loader,
        criterion_femba_tuar_tiny_2e5,
        device
    )

    history_femba_tuar_tiny_2e5.append({
        "method": "FEMBA_TUAR_tiny_two_stage_2e5",
        "stage": 2,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "head_lr": 1e-4,
        "backbone_lr": 2e-5,
        "seed": SEED
    })

    if val_loss < best_val_loss_femba_tuar_tiny_2e5:
        best_val_loss_femba_tuar_tiny_2e5 = val_loss

        torch.save({
            "method": "FEMBA_TUAR_tiny_two_stage_head1e4_backbone2e5_sbpdbp_loss",
            "model_state_dict": model_femba_tuar_tiny_2e5.state_dict(),
            "optimizer_state_dict": optimizer_femba_tuar_tiny_2e5_stage2.state_dict(),
            "best_val_loss": float(best_val_loss_femba_tuar_tiny_2e5),
            "stage1_epochs": num_epochs_stage1_2e5,
            "stage2_epoch": epoch + 1,
            "lr_stage1_head": 1e-4,
            "lr_stage2_backbone": 2e-5,
            "lr_stage2_head": 1e-4,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_femba_tuar_tiny_2e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2_2e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Stage 2 Epoch [1/20] | Train Loss: 1.243887 | Val Loss: 1.169784 saved
Stage 2 Epoch [2/20] | Train Loss: 1.109106 | Val Loss: 1.105346 saved
Stage 2 Epoch [3/20] | Train Loss: 1.055794 | Val Loss: 1.084859 saved
Stage 2 Epoch [4/20] | Train Loss: 1.021132 | Val Loss: 1.059700 saved
Stage 2 Epoch [5/20] | Train Loss: 0.991897 | Val Loss: 1.066865 
Stage 2 Epoch [6/20] | Train Loss: 0.974790 | Val Loss: 1.043402 saved
Stage 2 Epoch [7/20] | Train Loss: 0.959010 | Val Loss: 1.051179 
Stage 2 Epoch [8/20] | Train Loss: 0.940588 | Val Loss: 1.039662 saved
Stage 2 Epoch [9/20] | Train Loss: 0.929152 | Val Loss: 1.039751 
Stage 2 Epoch [10/20] | Train Loss: 0.918510 | Val Loss: 1.035277 saved
Stage 2 Epoch [11/20] | Train Loss: 0.910864 | Val Loss: 1.031355 saved
Stage 2 Epoch [12/20] | Train Loss: 0.900412 | Val Loss: 1.033857 
Stage 2 Epoch [13/20] | Train Loss: 0.894046 | Val Loss: 1.037349 
Stage 2 Epoch [14/20] | Train Loss: 0.884680 | Val Loss: 1.024946 saved
Stage 2 Epoch [15/20] | Tr

In [80]:
checkpoint_femba_tuar_tiny_2e5 = torch.load(
    best_model_path_femba_tuar_tiny_2e5,
    map_location="cpu"
)

model_femba_tuar_tiny_2e5_best = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
)

model_femba_tuar_tiny_2e5_best.load_state_dict(
    checkpoint_femba_tuar_tiny_2e5["model_state_dict"]
)

model_femba_tuar_tiny_2e5_best = model_femba_tuar_tiny_2e5_best.to(device)
model_femba_tuar_tiny_2e5_best.eval()

print("Loaded best FEMBA TUAR Tiny 2e-5 two-stage model")
print("Best Stage 2 epoch:", checkpoint_femba_tuar_tiny_2e5["stage2_epoch"])
print("Best val loss:", checkpoint_femba_tuar_tiny_2e5["best_val_loss"])
print("Stage 1 Head LR:", checkpoint_femba_tuar_tiny_2e5["lr_stage1_head"])
print("Stage 2 Backbone LR:", checkpoint_femba_tuar_tiny_2e5["lr_stage2_backbone"])
print("Stage 2 Head LR:", checkpoint_femba_tuar_tiny_2e5["lr_stage2_head"])

Loaded best FEMBA TUAR Tiny 2e-5 two-stage model
Best Stage 2 epoch: 20
Best val loss: 0.9639224094745407
Stage 1 Head LR: 0.0001
Stage 2 Backbone LR: 2e-05
Stage 2 Head LR: 0.0001


In [81]:
y_true_femba_tuar_tiny_2e5, y_pred_femba_tuar_tiny_2e5 = evaluate_model_waveform(
    model_femba_tuar_tiny_2e5_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_femba_tuar_tiny_2e5.shape)
print("y_pred:", y_pred_femba_tuar_tiny_2e5.shape)

print("true min/max:", y_true_femba_tuar_tiny_2e5.min(), y_true_femba_tuar_tiny_2e5.max())
print("pred min/max:", y_pred_femba_tuar_tiny_2e5.min(), y_pred_femba_tuar_tiny_2e5.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 55.72823 181.00392


In [82]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sbp_true_femba_tuar_tiny_2e5 = []
dbp_true_femba_tuar_tiny_2e5 = []
sbp_pred_femba_tuar_tiny_2e5 = []
dbp_pred_femba_tuar_tiny_2e5 = []

for true_sig, pred_sig in zip(y_true_femba_tuar_tiny_2e5, y_pred_femba_tuar_tiny_2e5):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_femba_tuar_tiny_2e5.append(sbp_t)
    dbp_true_femba_tuar_tiny_2e5.append(dbp_t)
    sbp_pred_femba_tuar_tiny_2e5.append(sbp_p)
    dbp_pred_femba_tuar_tiny_2e5.append(dbp_p)

sbp_true_femba_tuar_tiny_2e5 = np.array(sbp_true_femba_tuar_tiny_2e5)
dbp_true_femba_tuar_tiny_2e5 = np.array(dbp_true_femba_tuar_tiny_2e5)
sbp_pred_femba_tuar_tiny_2e5 = np.array(sbp_pred_femba_tuar_tiny_2e5)
dbp_pred_femba_tuar_tiny_2e5 = np.array(dbp_pred_femba_tuar_tiny_2e5)

sbp_mae_femba_tuar_tiny_2e5 = mean_absolute_error(
    sbp_true_femba_tuar_tiny_2e5,
    sbp_pred_femba_tuar_tiny_2e5
)

dbp_mae_femba_tuar_tiny_2e5 = mean_absolute_error(
    dbp_true_femba_tuar_tiny_2e5,
    dbp_pred_femba_tuar_tiny_2e5
)

sbp_rmse_femba_tuar_tiny_2e5 = np.sqrt(mean_squared_error(
    sbp_true_femba_tuar_tiny_2e5,
    sbp_pred_femba_tuar_tiny_2e5
))

dbp_rmse_femba_tuar_tiny_2e5 = np.sqrt(mean_squared_error(
    dbp_true_femba_tuar_tiny_2e5,
    dbp_pred_femba_tuar_tiny_2e5
))

sbp_r2_femba_tuar_tiny_2e5 = r2_score(
    sbp_true_femba_tuar_tiny_2e5,
    sbp_pred_femba_tuar_tiny_2e5
)

dbp_r2_femba_tuar_tiny_2e5 = r2_score(
    dbp_true_femba_tuar_tiny_2e5,
    dbp_pred_femba_tuar_tiny_2e5
)

avg_mae_femba_tuar_tiny_2e5 = (
    sbp_mae_femba_tuar_tiny_2e5 + dbp_mae_femba_tuar_tiny_2e5
) / 2

print("FEMBA TUAR Tiny Two-Stage 2e-5 Backbone Results")
print("===============================================")
print("Setting: Stage 1 Head 1e-4 10 epochs; Stage 2 Backbone 2e-5 Head 1e-4")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_femba_tuar_tiny_2e5:.3f}")
print(f"RMSE : {sbp_rmse_femba_tuar_tiny_2e5:.3f}")
print(f"R2   : {sbp_r2_femba_tuar_tiny_2e5:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_femba_tuar_tiny_2e5:.3f}")
print(f"RMSE : {dbp_rmse_femba_tuar_tiny_2e5:.3f}")
print(f"R2   : {dbp_r2_femba_tuar_tiny_2e5:.4f}")

print("\nTable format:")
print(f"{sbp_mae_femba_tuar_tiny_2e5:.2f}/{dbp_mae_femba_tuar_tiny_2e5:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_femba_tuar_tiny_2e5:.3f}")

FEMBA TUAR Tiny Two-Stage 2e-5 Backbone Results
Setting: Stage 1 Head 1e-4 10 epochs; Stage 2 Backbone 2e-5 Head 1e-4
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 5.973
RMSE : 8.315
R2   : 0.6875

DBP
MAE  : 2.915
RMSE : 4.448
R2   : 0.6035

Table format:
5.97/2.91

Average MAE:
4.444


In [83]:
results_femba_tuar_tiny_2e5 = pd.DataFrame([{
    "model": "FEMBA",
    "checkpoint": "TUAR tiny",
    "fine_tuning": "two-stage full fine-tuning",
    "loss": "Waveform + SBP + DBP loss",
    "stage1": "head only",
    "stage1_epochs": checkpoint_femba_tuar_tiny_2e5["stage1_epochs"],
    "stage1_head_lr": checkpoint_femba_tuar_tiny_2e5["lr_stage1_head"],
    "stage2": "full fine-tuning",
    "best_stage2_epoch": checkpoint_femba_tuar_tiny_2e5["stage2_epoch"],
    "stage2_backbone_lr": checkpoint_femba_tuar_tiny_2e5["lr_stage2_backbone"],
    "stage2_head_lr": checkpoint_femba_tuar_tiny_2e5["lr_stage2_head"],
    "seed": SEED,
    "best_val_loss": checkpoint_femba_tuar_tiny_2e5["best_val_loss"],
    "sbp_mae": sbp_mae_femba_tuar_tiny_2e5,
    "dbp_mae": dbp_mae_femba_tuar_tiny_2e5,
    "avg_mae": avg_mae_femba_tuar_tiny_2e5,
    "sbp_rmse": sbp_rmse_femba_tuar_tiny_2e5,
    "dbp_rmse": dbp_rmse_femba_tuar_tiny_2e5,
    "sbp_r2": sbp_r2_femba_tuar_tiny_2e5,
    "dbp_r2": dbp_r2_femba_tuar_tiny_2e5
}])

results_femba_tuar_tiny_2e5.to_csv(
    "results_femba_tuar_tiny_two_stage_head1e4_backbone2e5_sbpdbp_loss.csv",
    index=False
)

results_femba_tuar_tiny_2e5

,model,checkpoint,fine_tuning,loss,stage1,stage1_epochs,stage1_head_lr,stage2,best_stage2_epoch,stage2_backbone_lr,stage2_head_lr,seed,best_val_loss,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,FEMBA,TUAR tiny,two-stage full fine-tuning,Waveform + SBP + DBP loss,head only,10,0.0001,full fine-tuning,20,0.00002,0.0001,42,0.963922,5.972948,2.914515,4.443731,8.314906,4.448184,0.687543,0.603452


In [84]:
import gc
import torch

for name in list(globals().keys()):
    if (
        name.startswith("model_femba_tuar_tiny_disclr")
        or name.startswith("optimizer_femba_tuar_tiny_disclr")
        or name.startswith("checkpoint_femba_tuar_tiny_disclr")
    ):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Allocated GB: 0.487
Reserved GB: 0.543


In [85]:
model_femba_tuar_tiny_disclr = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
).to(device)

model_dict = model_femba_tuar_tiny_disclr.state_dict()
matched_state_disclr = {}

for ckpt_key, ckpt_value in state.items():
    candidates = [
        ckpt_key,
        "femba." + ckpt_key
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == ckpt_value.shape:
            matched_state_disclr[candidate] = ckpt_value
            break

missing, unexpected = model_femba_tuar_tiny_disclr.load_state_dict(
    matched_state_disclr,
    strict=False
)

print("Loaded tensors:", len(matched_state_disclr))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Loaded tensors: 42
Missing keys: 18
Unexpected keys: 0


In [86]:
for param in model_femba_tuar_tiny_disclr.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model_femba_tuar_tiny_disclr.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_femba_tuar_tiny_disclr.parameters()
)

print("Trainable params:", trainable_params)
print("Total params:", total_params)

Trainable params: 8575584
Total params: 8575584


In [87]:
criterion_femba_tuar_tiny_disclr = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_femba_tuar_tiny_disclr = torch.optim.AdamW(
    [
        {
            "params": model_femba_tuar_tiny_disclr.femba.parameters(),
            "lr": 2e-5
        },
        {
            "params": model_femba_tuar_tiny_disclr.regression_head.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-2
)

In [88]:
num_epochs_femba_tuar_tiny_disclr = 30
best_val_loss_femba_tuar_tiny_disclr = float("inf")

best_model_path_femba_tuar_tiny_disclr = "femba_tuar_tiny_full_ft_disclr_backbone2e5_head1e4_sbpdbp_seed42.pth"
history_femba_tuar_tiny_disclr = []

for epoch in range(num_epochs_femba_tuar_tiny_disclr):
    train_loss = train_one_epoch(
        model_femba_tuar_tiny_disclr,
        train_loader,
        optimizer_femba_tuar_tiny_disclr,
        criterion_femba_tuar_tiny_disclr,
        device
    )

    val_loss = validate_one_epoch(
        model_femba_tuar_tiny_disclr,
        val_loader,
        criterion_femba_tuar_tiny_disclr,
        device
    )

    history_femba_tuar_tiny_disclr.append({
        "method": "FEMBA_TUAR_tiny_full_ft_discriminative_lr",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 1e-4,
        "weight_decay": 1e-2,
        "seed": SEED,
        "waveform_weight": 1.0,
        "sbp_weight": 0.5,
        "dbp_weight": 0.5
    })

    if val_loss < best_val_loss_femba_tuar_tiny_disclr:
        best_val_loss_femba_tuar_tiny_disclr = val_loss

        torch.save({
            "method": "FEMBA_TUAR_tiny_full_ft_discriminative_lr",
            "model_state_dict": model_femba_tuar_tiny_disclr.state_dict(),
            "optimizer_state_dict": optimizer_femba_tuar_tiny_disclr.state_dict(),
            "best_val_loss": float(best_val_loss_femba_tuar_tiny_disclr),
            "epoch": epoch + 1,
            "backbone_lr": 2e-5,
            "head_lr": 1e-4,
            "weight_decay": 1e-2,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_femba_tuar_tiny_disclr)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_femba_tuar_tiny_disclr}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Epoch [1/30] | Train Loss: 1.307435 | Val Loss: 1.187412 saved
Epoch [2/30] | Train Loss: 1.124180 | Val Loss: 1.106940 saved
Epoch [3/30] | Train Loss: 1.060351 | Val Loss: 1.082978 saved
Epoch [4/30] | Train Loss: 1.023147 | Val Loss: 1.086963 
Epoch [5/30] | Train Loss: 0.993426 | Val Loss: 1.066012 saved
Epoch [6/30] | Train Loss: 0.972797 | Val Loss: 1.047820 saved
Epoch [7/30] | Train Loss: 0.951140 | Val Loss: 1.043213 saved
Epoch [8/30] | Train Loss: 0.931344 | Val Loss: 1.047128 
Epoch [9/30] | Train Loss: 0.910934 | Val Loss: 1.017684 saved
Epoch [10/30] | Train Loss: 0.892327 | Val Loss: 0.999654 saved
Epoch [11/30] | Train Loss: 0.867204 | Val Loss: 0.985860 saved
Epoch [12/30] | Train Loss: 0.839381 | Val Loss: 0.957997 saved
Epoch [13/30] | Train Loss: 0.801866 | Val Loss: 0.916397 saved
Epoch [14/30] | Train Loss: 0.760281 | Val Loss: 0.885020 saved
Epoch [15/30] | Train Loss: 0.721001 | Val Loss: 0.839375 saved
Epoch [16/30] | Train Loss: 0.681249 | Val Loss: 0.812756 s

In [89]:
checkpoint_femba_tuar_tiny_disclr = torch.load(
    best_model_path_femba_tuar_tiny_disclr,
    map_location="cpu"
)

model_femba_tuar_tiny_disclr_best = FEMBA_ECG_ABP(
    seq_length=625,
    num_channels=22,
    output_len=625,
    embed_dim=embed_dim,
    num_blocks=num_blocks
)

model_femba_tuar_tiny_disclr_best.load_state_dict(
    checkpoint_femba_tuar_tiny_disclr["model_state_dict"]
)

model_femba_tuar_tiny_disclr_best = model_femba_tuar_tiny_disclr_best.to(device)
model_femba_tuar_tiny_disclr_best.eval()

print("Loaded best FEMBA TUAR Tiny Full FT + Discriminative LR model")
print("Best epoch:", checkpoint_femba_tuar_tiny_disclr["epoch"])
print("Best val loss:", checkpoint_femba_tuar_tiny_disclr["best_val_loss"])
print("Backbone LR:", checkpoint_femba_tuar_tiny_disclr["backbone_lr"])
print("Head LR:", checkpoint_femba_tuar_tiny_disclr["head_lr"])

Loaded best FEMBA TUAR Tiny Full FT + Discriminative LR model
Best epoch: 30
Best val loss: 0.5794791160645646
Backbone LR: 2e-05
Head LR: 0.0001


In [90]:
y_true_femba_tuar_tiny_disclr, y_pred_femba_tuar_tiny_disclr = evaluate_model_waveform(
    model_femba_tuar_tiny_disclr_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_femba_tuar_tiny_disclr.shape)
print("y_pred:", y_pred_femba_tuar_tiny_disclr.shape)

print("true min/max:", y_true_femba_tuar_tiny_disclr.min(), y_true_femba_tuar_tiny_disclr.max())
print("pred min/max:", y_pred_femba_tuar_tiny_disclr.min(), y_pred_femba_tuar_tiny_disclr.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 48.30683 176.53012


In [91]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sbp_true_femba_tuar_tiny_disclr = []
dbp_true_femba_tuar_tiny_disclr = []
sbp_pred_femba_tuar_tiny_disclr = []
dbp_pred_femba_tuar_tiny_disclr = []

for true_sig, pred_sig in zip(y_true_femba_tuar_tiny_disclr, y_pred_femba_tuar_tiny_disclr):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_femba_tuar_tiny_disclr.append(sbp_t)
    dbp_true_femba_tuar_tiny_disclr.append(dbp_t)
    sbp_pred_femba_tuar_tiny_disclr.append(sbp_p)
    dbp_pred_femba_tuar_tiny_disclr.append(dbp_p)

sbp_true_femba_tuar_tiny_disclr = np.array(sbp_true_femba_tuar_tiny_disclr)
dbp_true_femba_tuar_tiny_disclr = np.array(dbp_true_femba_tuar_tiny_disclr)
sbp_pred_femba_tuar_tiny_disclr = np.array(sbp_pred_femba_tuar_tiny_disclr)
dbp_pred_femba_tuar_tiny_disclr = np.array(dbp_pred_femba_tuar_tiny_disclr)

sbp_mae_femba_tuar_tiny_disclr = mean_absolute_error(
    sbp_true_femba_tuar_tiny_disclr,
    sbp_pred_femba_tuar_tiny_disclr
)

dbp_mae_femba_tuar_tiny_disclr = mean_absolute_error(
    dbp_true_femba_tuar_tiny_disclr,
    dbp_pred_femba_tuar_tiny_disclr
)

sbp_rmse_femba_tuar_tiny_disclr = np.sqrt(mean_squared_error(
    sbp_true_femba_tuar_tiny_disclr,
    sbp_pred_femba_tuar_tiny_disclr
))

dbp_rmse_femba_tuar_tiny_disclr = np.sqrt(mean_squared_error(
    dbp_true_femba_tuar_tiny_disclr,
    dbp_pred_femba_tuar_tiny_disclr
))

sbp_r2_femba_tuar_tiny_disclr = r2_score(
    sbp_true_femba_tuar_tiny_disclr,
    sbp_pred_femba_tuar_tiny_disclr
)

dbp_r2_femba_tuar_tiny_disclr = r2_score(
    dbp_true_femba_tuar_tiny_disclr,
    dbp_pred_femba_tuar_tiny_disclr
)

avg_mae_femba_tuar_tiny_disclr = (
    sbp_mae_femba_tuar_tiny_disclr + dbp_mae_femba_tuar_tiny_disclr
) / 2

print("FEMBA TUAR Tiny Full FT + Discriminative LR Results")
print("===================================================")
print("Setting: Backbone LR 2e-5; Head LR 1e-4")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_femba_tuar_tiny_disclr:.3f}")
print(f"RMSE : {sbp_rmse_femba_tuar_tiny_disclr:.3f}")
print(f"R2   : {sbp_r2_femba_tuar_tiny_disclr:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_femba_tuar_tiny_disclr:.3f}")
print(f"RMSE : {dbp_rmse_femba_tuar_tiny_disclr:.3f}")
print(f"R2   : {dbp_r2_femba_tuar_tiny_disclr:.4f}")

print("\nTable format:")
print(f"{sbp_mae_femba_tuar_tiny_disclr:.2f}/{dbp_mae_femba_tuar_tiny_disclr:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_femba_tuar_tiny_disclr:.3f}")

FEMBA TUAR Tiny Full FT + Discriminative LR Results
Setting: Backbone LR 2e-5; Head LR 1e-4
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 5.796
RMSE : 8.075
R2   : 0.7053

DBP
MAE  : 3.082
RMSE : 4.623
R2   : 0.5717

Table format:
5.80/3.08

Average MAE:
4.439


In [92]:
results_femba_tuar_tiny_disclr = pd.DataFrame([{
    "model": "FEMBA",
    "checkpoint": "TUAR tiny",
    "fine_tuning": "full fine-tuning with discriminative LR",
    "loss": "Waveform + SBP + DBP loss",
    "backbone_lr": checkpoint_femba_tuar_tiny_disclr["backbone_lr"],
    "head_lr": checkpoint_femba_tuar_tiny_disclr["head_lr"],
    "seed": SEED,
    "best_epoch": checkpoint_femba_tuar_tiny_disclr["epoch"],
    "best_val_loss": checkpoint_femba_tuar_tiny_disclr["best_val_loss"],
    "sbp_mae": sbp_mae_femba_tuar_tiny_disclr,
    "dbp_mae": dbp_mae_femba_tuar_tiny_disclr,
    "avg_mae": avg_mae_femba_tuar_tiny_disclr,
    "sbp_rmse": sbp_rmse_femba_tuar_tiny_disclr,
    "dbp_rmse": dbp_rmse_femba_tuar_tiny_disclr,
    "sbp_r2": sbp_r2_femba_tuar_tiny_disclr,
    "dbp_r2": dbp_r2_femba_tuar_tiny_disclr
}])

results_femba_tuar_tiny_disclr.to_csv(
    "results_femba_tuar_tiny_full_ft_disclr_backbone2e5_head1e4_sbpdbp_loss.csv",
    index=False
)

results_femba_tuar_tiny_disclr

,model,checkpoint,fine_tuning,loss,backbone_lr,head_lr,seed,best_epoch,best_val_loss,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,FEMBA,TUAR tiny,full fine-tuning with discriminative LR,Waveform + SBP + DBP loss,0.00002,0.0001,42,30,0.579479,5.795768,3.081854,4.438811,8.075192,4.62289,0.705299,0.57169
